# 01. Customer Support on Twitter: Data Exploration

This notebook provides interactive exploration of the **Customer Support on Twitter** dataset (`twcs.csv`).
It demonstrates:
1. Memory-efficient chunked loading
2. Top 10 brands by volume
3. Multi-turn thread reconstruction
4. Domain-specific text cleaning
5. Customer message sampling for intent taxonomy design

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on Python path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.data_loader import DEFAULT_DATA_PATH, check_dataset_exists, stream_twcs_dataset
from src.thread_builder import reconstruct_threads
from src.text_cleaner import clean_tweet_text

print(f"Project root: {root_dir}")
print(f"Raw dataset path: {DEFAULT_DATA_PATH}")
print(f"Dataset exists: {check_dataset_exists(root_dir / DEFAULT_DATA_PATH)}")

## 1. Inspect First Few Rows of the Dataset

In [ ]:
# Load a small sample (first 1,000 rows)
from src.data_loader import load_twcs_dataset

sample_df = load_twcs_dataset(root_dir / DEFAULT_DATA_PATH, nrows=1000)
sample_df.head(10)

## 2. Reconstruct Sample Conversation Threads

In [ ]:
# Reconstruct threads from the sample
threads = reconstruct_threads(sample_df)
print(f"Reconstructed {len(threads)} conversation threads from sample.")

# Inspect a multi-turn thread
multi_threads = [t for t in threads if t['total_turns'] >= 2]
if multi_threads:
    example = multi_threads[0]
    print(f"\nConversation ID: {example['conversation_id']} ({example['total_turns']} turns):")
    for turn in example['turns']:
        role_tag = f"[{turn['role'].upper()} - @{turn['author_id']}]"
        print(f"  Turn {turn['turn_index']} {role_tag}: {turn['text_clean']}")

## 3. Test Text Cleaning Transformations

In [ ]:
test_tweets = [
    "@AmazonHelp My package #1234 hasn't arrived yet! 😡 Track here: https://t.co/xyz123 &amp; please help!",
    "@AppleSupport iPhone battery dropping from 100% to 20% in 1 hour after iOS update!! UNACCEPTABLE.",
    "@SpotifyCares Offline songs won't play when I have no wifi 🤬 http://spotify.com/help"
]

for t in test_tweets:
    cleaned = clean_tweet_text(t, brand_handle="AmazonHelp")
    print(f"ORIGINAL: {t}")
    print(f"CLEANED:  {cleaned}\n")